# 1.1 · PD training

*1. Experiment 1 · notebook 1.1 of the story.* ← [0.3 · the LGD prior](<../0. General/0.3_prior_visualisation_lgd.ipynb>) · [1.2 · LGD training](<1.2_lgd_training.ipynb>) →

**Experiment 1 asks: which prior?** A nano-scale TabICL model is trained **from scratch** on each
prior for 12,500 steps of 64 tasks — the same compute for every arm, which is the basis of the
comparison (`papers/2026/02_Qu_TabICLv2` §4.1). Three levers are crossed: the **credit fraction**
(the share of each batch drawn from our prior: 0 — the control, exactly TabICL's own prior — 0.5
or 1), the **filter mode** (`off`, `tabicl`, `banded`) and the **intensity** of our prior
(the Vasicek asset correlation ρ, mild [0.03, 0.12] or aggressive [0.12, 0.30]). That makes 15 priors, each trained with 3 seeds: 45 arms, all of which finished.

**What this notebook is — and is not.** While it trains, every arm is scored every 625 steps
(40,000 tasks) on the development split of the real datasets: 512 context rows, up to 2,000 scored.
That monitoring shows how training went; it does not decide. The decision is the benchmark in
[1.3 · PD results](<1.3_pd_results.ipynb>), where the prior is chosen on the development datasets and reported on the untouched
holdout (`docs/EXPERIMENTAL_DESIGN.md` §5).

**Three rules decide what every curve averages.** Only **finished** arms enter an average (an
unfinished arm stopping mid-curve would make it jump); only **development** datasets that every arm
scored enter it (the holdout is never averaged); and a group's curve is drawn only over the steps
**all** its arms logged (A2 explains why some logs start late).

**The order.** **A** — is the run sound: which arms exist and finished, what each was scored on,
what each cost, whether each learned. **B** — what the development monitoring says about each lever.
**C** — whether it holds on every dataset, outside credit and on every metric. **D** — every arm on
its own.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib
ROOT = pathlib.Path.cwd()
# Walk up to the repository root — the notebook may be opened from its chapter folder, from
# notebooks/, or from the root — then work FROM the root, so relative paths (config/...) resolve
# exactly as under `python -m src.utils.run_notebooks`.
while not (ROOT / "src" / "visualize").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import training_plots, figures, style, literature

style.apply()
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK = "pd"
EXP = "exp1"
# Reads output/manifests/ and output/results/ — whatever the runs have written so far, so a
# partial sweep still renders. Constructing the saver clears THIS notebook's figure folder only.
FIGS = figures.FigureSaver("1.1_pd_training")

## A · Is the run sound?

Before any number is read: which arms exist and finished, what each was scored on, what each cost,
and whether each one learned.

### A1 · Which arms exist, and how did each finish?

**What it shows.** Every scheduled arm in one grid: one row per configuration, grouped by credit fraction, one column per seed. A number is the arm's final development ROC-AUC — the mean over `german` and `myhom`; a grey cell has no development score.

**Why it matters.** A reader has to know which arms exist, which finished, and whether a block of rows or one seed stands out, before any comparison means anything: an effect shared by one seed moves every comparison that is not averaged over seeds.

**What it says.** All 45 arms finished all 12,500 steps. The cf = 1 block is visibly lower than the other two. Seed 2 scores highest of the three seeds in 12 of the 15 configurations — the seed fixes both the initialisation and which rows the monitor samples — which is why every comparison below averages over seeds.

In [ ]:
FIGS.save(training_plots.sweep_map(TASK, exp=EXP), "sweep_map",
    caption="Final development ROC-AUC of every scheduled PD arm as a heatmap, one row per configuration grouped by credit fraction and one column per seed; grey cells have no development score.");

### A2 · What was each arm scored on?

**What it shows.** One row per real dataset (development first, then holdout), one column per arm in the sweep map's order. A cell's colour says what that arm's score on that dataset may do: enter a development mean, exist without being averaged (not every arm carries it), be a holdout score (never averaged), or be a monitor that logged only missing values.

**Why it matters.** Arms were monitored under two protocols. Before 23-09-2026 the monitor scored "the four smallest" real datasets whatever their role, holdout ones included; since then, development datasets only. Averaging a holdout score into "which prior is best" would select on the test set, so this map fixes exactly which cells every curve in Part B is built from.

**What it says.** Every development mean in this notebook rests on two datasets, `german` and `myhom`, the only development datasets every arm scored. The 36 arms monitored before the protocol change also scored the holdout datasets `hmeq` and `thomas` — shown in C1, never averaged. `gmsc`, `lendingclub` and `taiwan_creditcard` were scored only by the nine arms restarted under the new protocol.

In [ ]:
FIGS.save(training_plots.monitoring_coverage(TASK, exp=EXP), "monitoring_coverage",
    caption="Monitoring coverage of every PD arm: one row per real dataset labelled development or holdout, one column per arm grouped by credit fraction, each cell coloured by whether its score enters the development mean, is not carried by every arm, is holdout, holds only missing values or was not monitored.");

### A3 · What did each arm cost?

**What it shows.** Each arm's training speed — its median steps per second over the run — grouped by filter mode, and the hours the full run takes at that speed; the bar is the group median, points are coloured and shaped by credit fraction.

**Why it matters.** TabICL generates its training tasks on the CPU while the GPU trains. A filter that throws most generated tasks away makes every batch cost several generations — a cost in walltime and VSC credits that never shows in the loss.

**What it says.** `off` and `tabicl` arms train at about 0.63–0.64 steps per second, 5–6 hours for the run. `banded` arms run at a median of 0.12 steps per second — about 29 hours — and the more of our prior an arm uses, the slower (cf = 0 about 0.23, cf = 1 about 0.08–0.11): the band keeps only a small share of our prior's tasks (0.2 C1). GPU utilisation reads 88–94 % in every arm, `banded` ones included, so the telemetry does not show where the extra time goes.

In [ ]:
FIGS.save(training_plots.throughput(TASK, exp=EXP), "throughput",
    caption="Training speed per PD arm in steps per second (left) and the implied hours for the full run (right), grouped by filter mode; points are arms coloured and shaped by credit fraction and bars are group medians with their value.");

### A4 · Did every arm learn?

**What it shows.** The training loss of every arm against the step — cross-entropy on the arm's own synthetic batch, as a rolling mean over three logged points (each logged value is one batch's loss) — with the median of the finished arms of each credit fraction drawn bold.

**Why it matters.** The loss is the one quantity every arm optimises directly, so divergence or a stall shows here first; TabICLv2's ablations warn that a prior can make training diverge (`papers/2026/02_Qu_TabICLv2` §4.4). But it is computed on each arm's own prior, so its level says which prior an arm trains on — not how well it learned.

**What it says.** No arm diverges. The levels separate by prior — final medians of about 0.30 at cf = 0, 0.19 at cf = 0.5 and 0.07 at cf = 1 — because our prior's low base rates make cross-entropy small, not because those arms learned more; within each fraction the curves flatten early. A level is never compared across fractions.

In [ ]:
FIGS.save(training_plots.training_loss(TASK, exp=EXP), "training_loss",
    caption="Training loss against optimisation step for every PD arm, a rolling mean over three logged batches coloured by credit fraction, with the median of each credit fraction bold.");

### A5 · Is every part of the network learning?

**What it shows.** Left, each block's gradient norm; right, the gradient divided by the block's weight norm — how far one step moves the block relative to its size. Both are averaged over all arms, on logarithmic axes. The telemetry records the column encoder, the row encoder and the ICL blocks; the prediction head is not logged.

**Why it matters.** A block whose gradient sits orders of magnitude below the others is effectively frozen, and the loss cannot say which one.

**What it says.** All three blocks keep learning. Every gradient falls steeply over the first 1,500 steps and then holds; the gradient-to-weight ratios settle at about 3 × 10⁻³ (column encoder), 1.7 × 10⁻³ (row encoder) and 5 × 10⁻⁴ (ICL blocks) — within an order of magnitude of each other, none collapsing.

In [ ]:
FIGS.save(training_plots.gradient_health(TASK, exp=EXP), "gradient_health",
    caption="Mean per-block gradient L2 norm (left) and ratio of gradient norm to weight norm (right) against training step on logarithmic axes, one line per architecture block (column encoder, row encoder, ICL blocks), averaged over all arms.");

## B · What does the development monitoring say?

The development ROC-AUC over training and at its end, lever by lever. It guides the eye; the benchmark in
[1.3 · PD results](<1.3_pd_results.ipynb>) decides.

### B1 · Development ROC-AUC over training

**What it shows.** The development ROC-AUC of every arm against the step, averaged over `german` and `myhom`, in its credit-fraction colour, with the mean of each fraction drawn bold.

**Why it matters.** The first look at how models trained on each prior do on real credit data they never trained on — and whether the fractions separate early, late or not at all.

**What it says.** The fractions separate at once and stay separated: cf = 0 and cf = 0.5 rise together to about 0.67, while cf = 1 starts near 0.58 and ends about 0.025 below them. The cf = 0.5 mean starts only at about step 4,000, because three of its arms' logs start there (A2).

In [ ]:
FIGS.save(training_plots.metric_over_training(TASK, exp=EXP), "metric_over_training",
    caption="Development ROC-AUC averaged over the development datasets every arm scored, against training step; one line per arm coloured by credit fraction, with the mean of each credit fraction bold.");

### B2 · Credit prior versus the control

**What it shows.** Credit-prior arms (any credit fraction above 0) against the control arms (credit fraction 0 — TabICL's own prior): the median over the finished arms of each group, with the inter-quartile band.

**Why it matters.** The experiment's comparison, as a curve. A gap narrower than the bands is not read as an effect.

**What it says.** The control leads from the start and stays ahead — about 0.672 against 0.662 at the end — with overlapping bands. The credit group mixes cf = 0.5, which matches the control, with cf = 1, which trails it (B1): the fraction matters more than 'credit or not'. The credit curve starts at about step 4,000 for the reason given in B1.

In [ ]:
FIGS.save(training_plots.credit_vs_control_over_training(TASK, exp=EXP), "credit_vs_control_over_training",
    caption="Development ROC-AUC against training step for credit-prior arms and control arms: median lines with shaded inter-quartile bands over the finished arms of each group.");

### B3 · Which lever moves the score?

**What it shows.** One panel per swept lever (credit fraction, filter mode and intensity), one mean curve per value over the finished arms that share it, all panels on one y axis.

**Why it matters.** The size of each lever's effect, side by side: a lever whose curves lie on top of each other does nothing, whatever the others do.

**What it says.** Only the credit fraction has a clear effect: cf = 1 ends about 0.025 below the other two. `off` and `tabicl` coincide, `banded` runs slightly lower and starts late (its restarted logs), and the two intensities lie on top of each other.

In [ ]:
FIGS.save(training_plots.metric_by_levers(TASK, exp=EXP), "metric_by_levers",
    caption="Development ROC-AUC against training step, one panel per swept lever with one mean line per lever value over the finished arms sharing it, all panels on a shared vertical axis.");

### B4 · Final score, lever by lever

**What it shows.** Every finished arm's final development score, grouped by each lever in turn: points coloured and shaped by credit fraction, a bar at each group mean, one shared y axis.

**Why it matters.** Final scores as a screen, one lever at a time. The sweep is factorial, so a lever's groups can differ merely because they contain different credit fractions; B5 separates the two.

**What it says.** The credit fraction separates the groups: means of 0.672 (cf = 0), 0.669 (cf = 0.5) and 0.646 (cf = 1). In every filter and intensity group the cf = 1 points cluster lowest, though single arms overlap with the other fractions', and those groups' means differ by 0.01 at most.

In [ ]:
FIGS.save(training_plots.final_metric_by_lever(TASK, exp=EXP), "final_metric_by_lever",
    caption="Final development ROC-AUC of every finished arm as points coloured and shaped by credit fraction, one panel per swept lever, with a horizontal bar at each group mean.");

### B5 · Each lever within each credit fraction

**What it shows.** Each lever other than the credit fraction, drawn within each fraction: points are finished arms, lines join the per-value means, whiskers are ±1 standard deviation over seeds. A lever the control does not have appears as the control's band (its mean ± 1 SD) instead.

**Why it matters.** Pooling a lever over fractions mixes the fraction's larger effect into it; within one fraction, a real effect of the other levers has nowhere to hide.

**What it says.** Within a fraction, the filter moves the mean by at most about 0.015 (cf = 0.5: `banded` 0.660 against `tabicl` 0.675) and the intensity by under 0.01 — less than the seed spread in most cells. The cf = 1 series lies below the control's band at every value of both levers.

In [ ]:
FIGS.save(training_plots.lever_interaction(TASK, exp=EXP), "lever_interaction",
    caption="Final development ROC-AUC against each lever other than credit fraction, one series per credit fraction: points are finished arms, lines join the per-value means, whiskers show one standard deviation over seeds, and the grey band is the control mean plus or minus one standard deviation.");

### B6 · Effect against seed noise

**What it shows.** Every configuration's seeds on one axis, best configuration at the top: points are seeds, the line their range, the tick their mean; the dashed line is the control's mean.

**Why it matters.** A difference between two configurations is only worth reading where it clearly exceeds the spread between one configuration's own seeds — the reason Experiment 1 runs three.

**What it says.** The seeds of one configuration span about 0.02 (median range 0.019) — as large as every difference between cf = 0 and cf = 0.5 configurations. Only the cf = 1 gap clearly exceeds it: its six configurations take the bottom six places.

In [ ]:
FIGS.save(training_plots.seed_spread(TASK, exp=EXP), "seed_spread",
    caption="Final development ROC-AUC of every configuration's seeds as points on one horizontal axis, configurations sorted by their mean with the range and mean marked per row; the dashed line is the control mean.");

## C · Where does it hold?

A mean can be carried by one easy dataset, and a gain on credit can be bought with general ability.
These figures check both, and every metric the monitor records.

### C1 · Every monitored dataset

**What it shows.** One panel per real dataset the monitor scored, development first: the mean over the finished credit-prior arms and over the finished control arms that scored it. Each title gives the dataset's side of the split and how many finished arms scored it.

**Why it matters.** Development panels are what the means above are built from. Holdout panels are shown as the evidence they are and never enter a mean — they belong to the benchmark.

**What it says.** The control is ahead on four of the five development datasets — by about 0.02 on `myhom`, under 0.01 on `german`, `gmsc` and `lendingclub` — and the two coincide on `taiwan_creditcard`; `gmsc`, `lendingclub` and `taiwan_creditcard` rest on only the six finished arms that scored them (A2). On the holdout datasets the control leads too (about 0.025 on `hmeq`, 0.013 on `thomas`); holdout panels are evidence, not a mean.

In [ ]:
for _p in range(1, training_plots.per_dataset_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.per_dataset_curves(TASK, _p, exp=EXP), f"per_dataset_p{_p}",
        caption="Monitored ROC-AUC against training step, one panel per real dataset labelled with its side of the split and the number of finished arms that scored it, credit-prior arms against control arms.");

### C2 · Credit data against out-of-domain data

**What it shows.** The development score on the real credit datasets (solid) and on eight public classification suites from outside credit (dashed), each averaged separately over credit-prior and control arms.

**Why it matters.** Out-of-domain retention is a first-class axis of the design (`docs/EXPERIMENTAL_DESIGN.md` §5.3): a prior that lifts credit scores by making the model worse everywhere else has not made it better.

**What it says.** The out-of-domain suites are easy for every arm (ROC-AUC 0.98–1.0), and the credit arms lose a little there (0.983 against 0.997), as they do on the credit datasets (0.658 against 0.672).

In [ ]:
FIGS.save(training_plots.real_vs_ood(TASK, exp=EXP), "real_vs_ood",
    caption="Development ROC-AUC on the real-credit datasets (solid) and on the out-of-domain suites (dashed) against training step, averaged separately over credit-prior and control arms.");

### C3 · Every development metric

**What it shows.** Every metric the monitor records, one figure per group (for LGD: accuracy; ranking and calibration; intervals and boundary atoms), one panel per metric: the mean over finished credit-prior and control arms. A title carries the metric's goal — ↑ higher is better, ↓ lower is better, → a value it should equal, drawn dashed; a predicted boundary share is drawn against the true share in the data.

**Why it matters.** The PD monitor logs only ROC-AUC and PR-AUC; PR-AUC depends on the base rate, so it is compared between arms on the same datasets, never across them. The calibration metrics (Brier, log-loss, calibration slope) arrive with the benchmark.

**What it says.** PR-AUC tells the same story as ROC-AUC: the control is ahead (about 0.535 against 0.512 at the end).

In [ ]:
for _p in range(1, training_plots.metric_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.all_eval_metrics(TASK, _p, exp=EXP), f"eval_metrics_p{_p}",
        caption="Each logged development metric against training step, one panel per metric and one figure per metric group, credit-prior and control means over the finished arms; the title marks the improving direction or target value, drawn as a dashed line.");

## D · Every arm on its own

For the reader who wants to check one configuration rather than trust a mean.

### D1 · Every configuration, seed by seed

**What it shows.** One panel per configuration in the sweep map's order: its seeds as separate lines of the development score, all panels on shared axes.

**Why it matters.** An arm that behaved unlike its neighbours, or unlike its own other seeds, stands out here and disappears in every mean.

**What it says.** Seed 2 (dotted) lies on top in most panels — the seed effect of A1. The cf = 1 panels rise more steeply and end lower; the cf = 0.5 `banded` lines start late, because their logs were restarted (A2).

In [ ]:
for _p in range(1, training_plots.config_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.per_config(TASK, _p, exp=EXP), f"per_config_p{_p}",
        caption="Development ROC-AUC against training step, one panel per configuration in sweep-map order with one line per seed, all panels on shared axes.");

### D2 · The best arm and the worst

**What it shows.** The finished arms with the highest and the lowest final development score: training loss and development score side by side.

**Why it matters.** Did the worst arm fail to learn, or learn something that does not transfer? The loss answers the first question, the score the second.

**What it says.** The worst arm (cf1·banded·aggr·s0) has the lower training loss throughout and the lower real-data score (0.625 against 0.688): it fits its own prior's tasks well and transfers worst. The loss level reflects the prior, not the quality (A4).

In [ ]:
FIGS.save(training_plots.best_and_worst(TASK, exp=EXP), "best_and_worst",
    caption="Training loss (a rolling mean over three logged batches) and development ROC-AUC against training step for the best and the worst finished arm by final development score.");

## Summary

The sweep in text: the run (A), what the development monitoring says (B) and where it holds (C). Printed last, in the order of the sections above, so `output/All_Results.md` carries the
same story as this notebook, followed by the `tfm-library` sources it cites (pin `e5ce016`) and the
list of figures.

In [ ]:
print(training_plots.training_summary(TASK, exp=EXP))
print()
print(literature.references_md(["batch", "stage1_lr", "optimizer", "filter_rate_clf", "filter_pval", "filter_extratrees_n", "merton_vasicek", "basel_corp"]))
print()
print(FIGS.summary())